In [2]:
import os

import pandas as pd

import time

import numpy as np

import tensorflow as tf
import tensorflow_probability as tfp

import matplotlib.pyplot as plt

In [3]:
os.chdir("scripts/")
%run -i log_linear_model.py
%run -i mixed_membership_model.py
%run -i MVR.py
os.chdir("../")

2025-05-20 14:39:30.925868: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-20 14:39:31.319562: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-20 14:39:31.319626: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-20 14:39:31.328887: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-20 14:39:31.328948: I tensorflow/compile

In [5]:
n_list = [1000, 5000, 10000]

N = 712174

input_path =  "data/synthetic/"

one_n_j = tf.convert_to_tensor(np.load(input_path+"NY_one_n_j.npy"), dtype = tf.int32)
X_ij_full    = tf.convert_to_tensor(np.load(input_path+"synth_X_ij_full.npy"), dtype = tf.int32)
d_j = tf.reduce_sum(one_n_j, axis = 1)

all_cells = full_states(d_j).numpy()

df_dict = {}
xy_dict = {}
for n in n_list:

	X_ij    = np.load(input_path+"synth_X_ij_"+str(n)+".npy")

	batch_size = 2048
	tau_in_data = step_tau_with_data(X_ij, one_n_j, X_ij_full[n:N,:], batch_size, N)
	print(r"n="+str(n)+" with tau: ", tau_in_data.numpy())

	df = pd.DataFrame(X_ij)

	df_count = df
	df_count["count"] = fast_frequency(X_ij, one_n_j).numpy()

	df_dict[n] = df_count.drop_duplicates().reset_index(drop=True)

	membership_matrix = np.all(np.expand_dims(all_cells, axis = 1)==np.expand_dims(df_dict[n].values[:,:-1], axis = 0), axis = -1).astype(float)

	y = np.einsum("ij,j->i", membership_matrix, df_dict[n].values[:,-1])
	xy_dict[n] = np.concatenate((tf.ones((tf.shape(all_cells)[0],1)), all_cells, np.expand_dims(y,axis=-1)), axis = -1)
	# xy_dict[n] = np.concatenate((all_cells, np.expand_dims(y,axis=-1)), axis = -1)

n=1000 with tau:  4.0
n=5000 with tau:  28.0
n=10000 with tau:  66.0


In [6]:
for n in n_list:
	pi = n/N

	x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
	y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)

	beta_tf = tf.Variable(tf.zeros(x.shape[1]), dtype = tf.float32)

	optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

	start = time.time()
	loss_optim, beta_optim, log_linear_tau = log_linear_disclosure_risk(beta_tf, pi, x, y, optimizer, 2000)
	print(r"n="+str(n)+" with tau from log-linear: ", log_linear_tau.numpy())
	print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear:  10.974495
in  0.0026869110266367594
n=5000 with tau from log-linear:  55.28279
in  0.0022987226645151773
n=10000 with tau from log-linear:  110.75528
in  0.002303003470102946


In [11]:
for n in n_list:
	pi = n/N

	y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)
	x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
	x_interaction = x

	for i in range(1, x_interaction.shape[1]):

		for j in range(i+1, x_interaction.shape[1]):
			x_interaction = tf.concat((x_interaction, x[:,i:i+1]*x[:,j:j+1]), axis = -1)

	beta_tf = tf.Variable(tf.zeros(x_interaction.shape[1]), dtype = tf.float32)

	optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

	start = time.time()
	loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = log_linear_disclosure_risk(beta_tf, pi, x_interaction, y, optimizer, 2000)
	print(r"n="+str(n)+" with tau from log-linear with interaction: ", log_linear_tau_interaction.numpy())
	print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear with interaction:  19.223036
in  0.0023546996381547717
n=5000 with tau from log-linear with interaction:  96.9309
in  0.0024711985058254666
n=10000 with tau from log-linear with interaction:  190.33282
in  0.0025109212266074287


In [12]:
lambda_0_list = [tf.constant(1, dtype = tf.float32), tf.constant(10, dtype = tf.float32), tf.constant(100, dtype = tf.float32)]

for lambda_0 in lambda_0_list:

	for n in n_list:
		pi = n/N

		y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)
		x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
		x_interaction = x

		for i in range(1, x_interaction.shape[1]):

			for j in range(i+1, x_interaction.shape[1]):
				x_interaction = tf.concat((x_interaction, x[:,i:i+1]*x[:,j:j+1]), axis = -1)

		beta_tf = tf.Variable(tf.zeros(x_interaction.shape[1]), dtype = tf.float32)

		optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

		start = time.time()
		loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = penalized_log_linear_disclosure_risk(beta_tf, pi, x_interaction, y, lambda_0, optimizer, 2000)
		print(r"n="+str(n)+" with tau from log-linear with penalized interaction: ", log_linear_tau_interaction.numpy())
		print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear with penalized interaction:  11.537272
in  0.002810661064253913
n=5000 with tau from log-linear with penalized interaction:  57.529076
in  0.002742935882674323
n=10000 with tau from log-linear with penalized interaction:  113.807755
in  0.0026740574174457126
n=1000 with tau from log-linear with penalized interaction:  12.115989
in  0.0025633356968561808
n=5000 with tau from log-linear with penalized interaction:  57.50675
in  0.0025574833816952177
n=10000 with tau from log-linear with penalized interaction:  113.5698
in  0.002556595007578532
n=1000 with tau from log-linear with penalized interaction:  14.863818
in  0.0025612327125337387
n=5000 with tau from log-linear with penalized interaction:  60.831543
in  0.0025441928704579672
n=10000 with tau from log-linear with penalized interaction:  115.32646
in  0.002581514981057909


# Real data

In [16]:
n_list = [1000, 5000, 10000]

N = 712174

input_path =  "data/clean/"

one_n_j = tf.convert_to_tensor(np.load(input_path+"NY_one_n_j.npy"), dtype = tf.int32)
X_ij_full    = tf.convert_to_tensor(np.load(input_path+"NY_X_ij_full.npy"), dtype = tf.int32)
d_j = tf.reduce_sum(one_n_j, axis = 1)

all_cells = full_states(d_j).numpy()

df_dict = {}
xy_dict = {}
for n in n_list:

	X_ij    = np.load(input_path+"NY_X_ij_"+str(n)+".npy")

	batch_size = 2048
	tau_in_data = step_tau_with_data(X_ij, one_n_j, X_ij_full[n:N,:], batch_size, N)
	print(r"n="+str(n)+" with tau: ", tau_in_data.numpy())

	df = pd.DataFrame(X_ij)

	df_count = df
	df_count["count"] = fast_frequency(X_ij, one_n_j).numpy()

	df_dict[n] = df_count.drop_duplicates().reset_index(drop=True)

	membership_matrix = np.all(np.expand_dims(all_cells, axis = 1)==np.expand_dims(df_dict[n].values[:,:-1], axis = 0), axis = -1).astype(float)

	y = np.einsum("ij,j->i", membership_matrix, df_dict[n].values[:,-1])
	xy_dict[n] = np.concatenate((tf.ones((tf.shape(all_cells)[0],1)), all_cells, np.expand_dims(y,axis=-1)), axis = -1)
	# xy_dict[n] = np.concatenate((all_cells, np.expand_dims(y,axis=-1)), axis = -1)

n=1000 with tau:  9.0
n=5000 with tau:  27.0
n=10000 with tau:  53.0


In [17]:
for n in n_list:
	pi = n/N

	x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
	y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)

	beta_tf = tf.Variable(tf.zeros(x.shape[1]), dtype = tf.float32)

	optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

	start = time.time()
	loss_optim, beta_optim, log_linear_tau = log_linear_disclosure_risk(beta_tf, pi, x, y, optimizer, 2000)
	print(r"n="+str(n)+" with tau from log-linear: ", log_linear_tau.numpy())
	print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear:  12.509193
in  0.0024696592489878337
n=5000 with tau from log-linear:  49.60178
in  0.0025209870603349472
n=10000 with tau from log-linear:  85.466354
in  0.002564705014228821


In [18]:
for n in n_list:
	pi = n/N

	y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)
	x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
	x_interaction = x

	for i in range(1, x_interaction.shape[1]):

		for j in range(i+1, x_interaction.shape[1]):
			x_interaction = tf.concat((x_interaction, x[:,i:i+1]*x[:,j:j+1]), axis = -1)

	beta_tf = tf.Variable(tf.zeros(x_interaction.shape[1]), dtype = tf.float32)

	optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

	start = time.time()
	loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = log_linear_disclosure_risk(beta_tf, pi, x_interaction, y, optimizer, 2000)
	print(r"n="+str(n)+" with tau from log-linear with interaction: ", log_linear_tau_interaction.numpy())
	print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear with interaction:  12.328907
in  0.0026105015807681613
n=5000 with tau from log-linear with interaction:  52.75303
in  0.0026603917280832927
n=10000 with tau from log-linear with interaction:  95.13638
in  0.002479134202003479


In [19]:
lambda_0_list = [tf.constant(1, dtype = tf.float32), tf.constant(10, dtype = tf.float32), tf.constant(100, dtype = tf.float32)]

for lambda_0 in lambda_0_list:

	for n in n_list:
		pi = n/N

		y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)
		x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
		x_interaction = x

		for i in range(1, x_interaction.shape[1]):

			for j in range(i+1, x_interaction.shape[1]):
				x_interaction = tf.concat((x_interaction, x[:,i:i+1]*x[:,j:j+1]), axis = -1)

		beta_tf = tf.Variable(tf.zeros(x_interaction.shape[1]), dtype = tf.float32)

		optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

		start = time.time()
		loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = penalized_log_linear_disclosure_risk(beta_tf, pi, x_interaction, y, lambda_0, optimizer, 2000)
		print(r"n="+str(n)+" with tau from log-linear with penalized interaction: ", log_linear_tau_interaction.numpy())
		print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear with penalized interaction:  8.941475
in  0.0028839692142274644
n=5000 with tau from log-linear with penalized interaction:  57.483784
in  0.002842645777596368
n=10000 with tau from log-linear with penalized interaction:  97.23178
in  0.0029014359580145943
n=1000 with tau from log-linear with penalized interaction:  9.382984
in  0.0029433106051550972
n=5000 with tau from log-linear with penalized interaction:  55.066483
in  0.0028724808163113065
n=10000 with tau from log-linear with penalized interaction:  95.20505
in  0.0027754008769989014
n=1000 with tau from log-linear with penalized interaction:  7.3157477
in  0.002890774408976237
n=5000 with tau from log-linear with penalized interaction:  47.609207
in  0.00278277940220303
n=10000 with tau from log-linear with penalized interaction:  88.61246
in  0.0027148316303888956


In [20]:
lambda_0_list = [tf.constant(1000, dtype = tf.float32), tf.constant(5000, dtype = tf.float32), tf.constant(10000, dtype = tf.float32)]

for lambda_0 in lambda_0_list:

	for n in n_list:
		pi = n/N

		y = tf.convert_to_tensor(xy_dict[n][:,-1], dtype = tf.float32)
		x = tf.concat(tf.convert_to_tensor(xy_dict[n][:,:-1], dtype = tf.float32), axis = -1)
		x_interaction = x

		for i in range(1, x_interaction.shape[1]):

			for j in range(i+1, x_interaction.shape[1]):
				x_interaction = tf.concat((x_interaction, x[:,i:i+1]*x[:,j:j+1]), axis = -1)

		beta_tf = tf.Variable(tf.zeros(x_interaction.shape[1]), dtype = tf.float32)

		optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

		start = time.time()
		loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = penalized_log_linear_disclosure_risk(beta_tf, pi, x_interaction, y, lambda_0, optimizer, 2000)
		print(r"n="+str(n)+" with tau from log-linear with penalized interaction: ", log_linear_tau_interaction.numpy())
		print("in ", str((time.time()-start)/3600))

n=1000 with tau from log-linear with penalized interaction:  8.866949
in  0.0027938422229554917
n=5000 with tau from log-linear with penalized interaction:  24.380726
in  0.002854269478056166
n=10000 with tau from log-linear with penalized interaction:  64.07874
in  0.0029314884212281967
n=1000 with tau from log-linear with penalized interaction:  31.767815
in  0.0030751106474134655
n=5000 with tau from log-linear with penalized interaction:  40.569504
in  0.0029000186920166014
n=10000 with tau from log-linear with penalized interaction:  56.0027
in  0.0027422785096698336
n=1000 with tau from log-linear with penalized interaction:  52.78551
in  0.0027034042278925577
n=5000 with tau from log-linear with penalized interaction:  41.15643
in  0.00270714971754286
n=10000 with tau from log-linear with penalized interaction:  63.66558
in  0.0027531048324373033


# Real data with structural zeros

In [9]:
n_list = [1000]#[1000, 5000, 10000]

N = 953076

input_path =  "data/structural_zeros/"

one_n_j = tf.convert_to_tensor(np.load(input_path+"SZ_NY_one_n_j.npy"), dtype = tf.int32)
X_ij_full    = tf.convert_to_tensor(np.load(input_path+"SZ_NY_X_ij_full.npy"), dtype = tf.int32)
d_j = tf.reduce_sum(one_n_j, axis = 1)

all_cells = full_states(d_j).numpy()

for n in n_list:

	X_ij    = np.load(input_path+"SZ_NY_X_ij_"+str(n)+".npy")

	batch_size = 1024
	tau_in_data = step_tau_with_data(X_ij, one_n_j, X_ij_full[n:N,:], batch_size, N)
	print(r"n="+str(n)+" with tau: ", tau_in_data.numpy())

	df = pd.DataFrame(X_ij)

	df_count = df
	df_count["count"] = fast_frequency(X_ij, one_n_j).numpy()

	df_drop = df_count.drop_duplicates().reset_index(drop=True)

	membership_matrix_list = []
	y_list = []
	for i in range(40):
		# print(i)
		membership_matrix_batch = tf.reduce_all(tf.expand_dims(all_cells[int(256608/4)*i:int(256608/4)*(i+1)], axis = 1)==tf.expand_dims(tf.convert_to_tensor(df_drop.values[:,:-1], dtype = tf.int32), axis = 0), axis = -1)
		membership_matrix_list.append(membership_matrix_batch)

		y_batch = tf.einsum("ij,j->i", tf.cast(membership_matrix_batch, dtype = tf.float32), tf.convert_to_tensor(df_drop.values[:,-1], dtype = tf.float32))
		y_list.append(y_batch)

	pi = n/N

	x = tf.concat((tf.ones((tf.shape(all_cells)[0],1)), tf.convert_to_tensor(all_cells, dtype = tf.float32)), axis = -1)
	y = tf.concat(y_list, axis = 0)

	beta_tf = tf.Variable(tf.zeros(x.shape[1]), dtype = tf.float32)

	optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)

	start = time.time()
	loss_optim, beta_optim, log_linear_tau = log_linear_disclosure_risk(beta_tf, pi, x, y, optimizer, 2000)
	print(r"n="+str(n)+" with tau from log-linear: ", log_linear_tau.numpy())
	print("in ", str((time.time()-start)/3600))
	
	lambda_0_list = [tf.constant(1, dtype = tf.float32), tf.constant(10, dtype = tf.float32), tf.constant(100, dtype = tf.float32)]

	for lambda_0 in lambda_0_list:

		beta_tf = tf.Variable(tf.zeros(x.shape[1]), dtype = tf.float32)

		optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

		start = time.time()
		loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = penalized_log_linear_disclosure_risk(beta_tf, pi, x, y, lambda_0, optimizer, 2000)
		print(r"n="+str(n)+" with tau from log-linear with penalized interaction: ", log_linear_tau_interaction.numpy())
		print("in ", str((time.time()-start)/3600))

n=1000 with tau:  11.0
n=1000 with tau from log-linear:  170.09915
in  0.0026794695854187012
n=1000 with tau from log-linear with penalized interaction:  170.0888
in  0.0030578082137637666
n=1000 with tau from log-linear with penalized interaction:  169.11896
in  0.0030572528971566093
n=1000 with tau from log-linear with penalized interaction:  163.17638
in  0.003020896514256795


In [12]:
for n in n_list:

	X_ij    = np.load(input_path+"SZ_NY_X_ij_"+str(n)+".npy")

	batch_size = 1024
	tau_in_data = step_tau_with_data(X_ij, one_n_j, X_ij_full[n:N,:], batch_size, N)
	print(r"n="+str(n)+" with tau: ", tau_in_data.numpy())

	df = pd.DataFrame(X_ij)

	df_count = df
	df_count["count"] = fast_frequency(X_ij, one_n_j).numpy()

	df_drop = df_count.drop_duplicates().reset_index(drop=True)

	membership_matrix_list = []
	y_list = []
	for i in range(40):
		# print(i)
		membership_matrix_batch = tf.reduce_all(tf.expand_dims(all_cells[int(256608/4)*i:int(256608/4)*(i+1)], axis = 1)==tf.expand_dims(tf.convert_to_tensor(df_drop.values[:,:-1], dtype = tf.int32), axis = 0), axis = -1)
		membership_matrix_list.append(membership_matrix_batch)

		y_batch = tf.einsum("ij,j->i", tf.cast(membership_matrix_batch, dtype = tf.float32), tf.convert_to_tensor(df_drop.values[:,-1], dtype = tf.float32))
		y_list.append(y_batch)

	pi = n/N

	x = tf.concat((tf.ones((tf.shape(all_cells)[0],1)), tf.convert_to_tensor(all_cells, dtype = tf.float32)), axis = -1)
	y = tf.concat(y_list, axis = 0)
	
	lambda_0_list = [tf.constant(1000, dtype = tf.float32), tf.constant(10000, dtype = tf.float32), tf.constant(100000, dtype = tf.float32), tf.constant(100000, dtype = tf.float32)]

	for lambda_0 in lambda_0_list:

		beta_tf = tf.Variable(tf.zeros(x.shape[1]), dtype = tf.float32)

		optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

		start = time.time()
		loss_optim_interaction, beta_optim_interaction, log_linear_tau_interaction = penalized_log_linear_disclosure_risk(beta_tf, pi, x, y, lambda_0, optimizer, 2000)
		print(r"n="+str(n)+" with tau from log-linear with penalized interaction: ", log_linear_tau_interaction.numpy())
		print("in ", str((time.time()-start)/3600))

n=1000 with tau:  11.0
n=1000 with tau from log-linear with penalized interaction:  216.31615
in  0.0030905667940775553
n=1000 with tau from log-linear with penalized interaction:  181.77211
in  0.0029096989499198065
n=1000 with tau from log-linear with penalized interaction:  181.38924
in  0.0029224860668182373
n=1000 with tau from log-linear with penalized interaction:  181.38924
in  0.0029422112968232896
